## CDI Homeowners Non-Renewals by ZIP Code
Load CDI's ZIP-level policy counts (2015–2023) and calculate non-renewal rates

In [23]:
from pathlib import Path
import pandas as pd

raw = Path("../data/raw")

In [24]:
#list raw files
for f in sorted(raw.iterdir()):
    print(f.name)

cb_2020_us_county_500k.zip
cb_2020_us_zcta520_500k.zip
FHSZLRA251Allgdb
FHSZSRA_23_3
ZIP2015-2021.xlsx
ZIP2020-2023.xlsx


In [25]:
#preview 2015-2021
xl_old = pd.ExcelFile(raw / "ZIP2015-2021.xlsx")
print(xl_old.sheet_names)
pd.read_excel(xl_old, sheet_name = xl_old.sheet_names[0], header = None, nrows = 12)

['New Renew NonRenew']


,0,1,2,3,4,5,6
0,County,ZIP Code,Year,New,Renewed,Insured-Initiated Nonrenewed,Insurer-Initiated Nonrenewed
1,Los Angeles,90001,2015,734,5092,415,178
2,Los Angeles,90002,2015,807,5006,472,198
3,Los Angeles,90003,2015,1033,6115,612,305
4,Los Angeles,90004,2015,527,3875,400,128
5,Los Angeles,90005,2015,130,859,113,42
6,Los Angeles,90006,2015,314,1645,179,91
7,Los Angeles,90007,2015,283,1682,153,93
8,Los Angeles,90008,2015,528,4247,322,104
9,Los Angeles,90009,2015,0,3,0,0


In [26]:
#preview 2020-2023
xl_new = pd.ExcelFile(raw / "ZIP2020-2023.xlsx")
print(xl_new.sheet_names)
pd.read_excel(xl_new, sheet_name = xl_new.sheet_names[0], header = None, nrows = 12)

['New Renew Nonrenew']


,0,1,2,3,4,5
0,County,ZIP Code,Year,New,Renewed,Non-Renewed
1,,90000,2020,1,3,0
2,,90000,2021,1,0,3
3,,90000,2022,4,49,27
4,,90000,2023,0,25,18
5,Los Angeles,90001,2020,788,5539,645
6,Los Angeles,90001,2021,907,5566,708
7,Los Angeles,90001,2022,843,5614,762
8,Los Angeles,90001,2023,774,5569,727
9,Los Angeles,90002,2020,836,5518,625


The two files count nonrenewals differently. 2015-2021 splits nonrenewed into columns of "Insured-Initiated" and "Insurer-Initiated", while the 2020-2023 file has a single nonrenewed column. Zip 90000 also has no county.

In [27]:
#load both files
old = pd.read_excel(raw / "ZIP2015-2021.xlsx")
new = pd.read_excel(raw / "ZIP2020-2023.xlsx")

print(old.columns.tolist())
print(new.columns.tolist())

['County', 'ZIP Code', 'Year', 'New', 'Renewed', 'Insured-Initiated Nonrenewed', 'Insurer-Initiated Nonrenewed']
['County', 'ZIP Code', 'Year', 'New', 'Renewed', 'Non-Renewed']


In [28]:
#rename columns
old.columns = ["county", "zip", "year", "new", "renewed", "nonrenewed_insured", "nonrenewed_insurer"]
new.columns = ["county", "zip", "year", "new", "renewed", "nonrenewed"]

In [29]:
#convert ZIP code text and check
old["zip"] = old["zip"].astype(str)
new["zip"] = new["zip"].astype(str)

print(old["zip"].str.len().value_counts())
print(new["zip"].str.len().value_counts())

zip
5    15538
Name: count, dtype: int64
zip
5    8032
Name: count, dtype: int64


In [30]:
#last checks
print(old.shape)
print(old["year"].value_counts().sort_index())

(15538, 7)
year
2015    2476
2016    2339
2017    2220
2018    2120
2019    2103
2020    2154
2021    2126
Name: count, dtype: int64


In [31]:
print(new.shape)
print(new["year"].value_counts().sort_index())

(8032, 6)
year
2020    2113
2021    2041
2022    1945
2023    1933
Name: count, dtype: int64


The 2020-2021 counts don't match between files, so compare overlapping years and figure out what's going on.

In [32]:
#total non-renewals = insured-initiated + insurer-initiated
#we combine it so we can easily compare
old["nonrenewed"] = old["nonrenewed_insured"] + old["nonrenewed_insurer"]

In [33]:
#match ZIP and year on 2020-2021 from each file
overlap = old[old["year"].isin([2020, 2021])].merge(new[new["year"].isin([2020, 2021])],
    on = ["zip", "year"],
    how = "outer",
    suffixes = ("_old", "_new"),
    indicator = True,)
    
overlap["_merge"].value_counts()

_merge
both          3951
left_only      329
right_only     203
Name: count, dtype: int64

In [34]:
#do the numbers match?
both = overlap[overlap["_merge"] == "both"]

for col in ["new", "renewed", "nonrenewed"]:
    match_rate = (both[f"{col}_old"] == both[f"{col}_new"]).mean()
    print(f"{col}: {match_rate:.1%} of rows match")

new: 19.2% of rows match
renewed: 9.5% of rows match
nonrenewed: 13.0% of rows match


In [35]:
#compare statewide totals
cols = ["new", "renewed", "nonrenewed"]
print(old[old["year"].isin([2020, 2021])].groupby("year")[cols].sum())
print(new[new["year"].isin([2020, 2021])].groupby("year")[cols].sum())

          new  renewed  nonrenewed
year                              
2020  1057402  7631979      992227
2021  1107169  7621326     1118422
          new  renewed  nonrenewed
year                              
2020  1007422  7572809      898064
2021  1091089  7514564     1037767


The two files don't agree on the numbers, so we will treat them as separate. Since 2015-2021 directly measures insurers dropping homeowners (insurer-initiated vs insured-initiated non-renewals), we will use that file as our primary analysis!

In [36]:
#statewide non-renewal rate by year, per file
for name, df in [("old", old), ("new", new)]:
    t = df.groupby("year")[["renewed", "nonrenewed"]].sum()
    t["rate"] = t["nonrenewed"] / (t["renewed"] + t["nonrenewed"])

    print(name)
    print(t["rate"].round(3))

old
year
2015    0.108
2016    0.109
2017    0.109
2018    0.108
2019    0.119
2020    0.115
2021    0.128
Name: rate, dtype: float64
new
year
2020    0.106
2021    0.121
2022    0.106
2023    0.094
Name: rate, dtype: float64


Why does the statewide rate drop after 2021? Also observe: rates are flat at ~10.8–10.9% from 2015 to 2018, then jumps in 2019, right after the 2017 and 2018 wildfire seasons (notably Tubbs, Camp, Woolsey)

In [37]:
#CA ZIPs run 90001-96162 (we saw 90000 above).
#anything else is a placeholder or error, quick check
for name, df in [("old", old), ("new", new)]:
    odd = df[~df["zip"].between("90001", "96162")]
    print(name, sorted(odd["zip"].unique()), len(odd), "rows")

old ['78681', '85670', '90000', '96165', '96191', '96745', '96756', '96825', '97302', '97377', '97457', '97635', '97838', '98070', '98110', '98502', '99337'] 21 rows
new ['90000'] 4 rows


In [38]:
#combining both files with a source label
old_clean = old[["county", "zip", "year", "renewed", "nonrenewed", "nonrenewed_insurer"]].copy()
old_clean["source"] = "2015-2021 file"

new_clean = new[["county", "zip", "year", "renewed", "nonrenewed"]].copy()
new_clean["source"] = "2020-2023 file"

cdi = pd.concat([old_clean, new_clean], ignore_index = True)
print(cdi.shape)
cdi["source"].value_counts()


(23570, 7)


source
2015-2021 file    15538
2020-2023 file     8032
Name: count, dtype: int64

In [39]:
#calculate the rates
cdi["up_for_renewal"] = cdi["renewed"] + cdi["nonrenewed"]
cdi["nonrenewal_rate"] = cdi["nonrenewed"] / cdi["up_for_renewal"]
cdi["insurer_nonrenewal_rate"] = cdi["nonrenewed_insurer"] / cdi["up_for_renewal"]

cdi[["nonrenewal_rate", "insurer_nonrenewal_rate"]].describe()

,nonrenewal_rate,insurer_nonrenewal_rate
count,23060.000000,15089.000000
mean,0.125796,0.036528
std,0.136831,0.091799
min,0.000000,0.000000
25%,0.084943,0.011897
50%,0.106287,0.020339
75%,0.130952,0.032538
max,1.000000,1.000000


There are zip codes from other states we need to exclude, whatever the reason.  Remove and record: 25 rows dropped, 144 policies up for renewal.

In [40]:
#ZIP drop rows
in_ca = cdi["zip"].between("90001", "96162")
print(f"Dropping {(~in_ca).sum()} rows "
      f"({cdi.loc[~in_ca, 'up_for_renewal'].sum():,} policies up for renewal)")

cdi = cdi[in_ca].copy()
print(cdi.shape)

Dropping 25 rows (144 policies up for renewal)
(23545, 10)


In [41]:
#we want a minimum volume for a reliable rate
total = cdi["up_for_renewal"].sum()
for t in [25, 50, 100, 250]:
    small = cdi["up_for_renewal"] < t
    print(f"under {t}: {small.mean():.1%} of rows, "
          f"{cdi.loc[small, 'up_for_renewal'].sum() / total:.2%} of policies")

under 25: 20.8% of rows, 0.02% of policies
under 50: 23.5% of rows, 0.04% of policies
under 100: 27.3% of rows, 0.12% of policies
under 250: 34.6% of rows, 0.42% of policies


In [42]:
#flags the rows, doesnt delete. set cutoff at < 100
cdi["low_volume"] = cdi["up_for_renewal"] < 100
cdi["low_volume"].value_counts()

low_volume
False    17123
True      6422
Name: count, dtype: int64

In [43]:
#save
clean = Path("../data/clean")
clean.mkdir(exist_ok = True)
cdi.to_csv(clean / "cdi_nonrenewals_by_zip_year.csv", index = False)

In [44]:
#check
check = pd.read_csv(clean / "cdi_nonrenewals_by_zip_year.csv", dtype = {"zip": str})
print(check.shape)
print(check["zip"].str.len().unique())

(23545, 11)
[5]


The clean file has one row per ZIP code, year, and source file (23,545 rows), with the two CDI files kept as separate series since they don't agree for 2020–2021. I dropped 25 rows with ZIP codes outside California (144 policies) and flagged ZIP-years with fewer than 100 policies up for renewal as low volume, which covers 27% of rows but only 0.12% of policies.